# UV Setup Tutorial: End-to-End Mini Project

This hands-on notebook guides you through the full Astral `uv` toolchain to create a small, illustrative Python project using `pyproject.toml` for configuration. You will:

- Initialize a project with `uv init`
- Manage dependencies with `uv add`, `uv sync`, and `uv lock`
- Create a small package and CLI entry point
- Add tests and linting
- Build distributions and manage cache

Run each cell in order. Shell commands are shown using `` or notebook magics.


## Prerequisites and Installation

- Ensure you have Python 3.8+ available.
- Confirm you can run shell commands from this notebook (Jupyter supports `` for shell and `%cd` for directory changes).

You can install `uv` via one of the following methods:

1) pip (in a user scope or virtual environment):

```bash
python -m pip install -U uv
```

2) pipx (recommended for isolated CLI installs):

```bash
pipx install uv
```

3) Official installer script (fast, standalone):

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

After installation, verify `uv` is available:

```bash
uv --version
```

## 1) Initialize a new project
This will create a new directory with a `pyproject.toml`, a virtual environment, and helpful defaults.

We will use `mini_uv_project` as the example name.

Run the following to initialize:

```bash
uv init mini_uv_project
```

If you prefer, you can initialize in-place (current directory) by omitting the name:

```bash
uv init
```


In [ ]:
# Create the example project directory using uv
uv init mini_uv_project | cat

## 2) Add and manage dependencies
We will add one runtime dependency (`requests`) and one development dependency (`pytest`). `uv add` updates `pyproject.toml` and `uv.lock` (after locking) and `uv sync` installs them into the environment.

Commands:

```bash
uv add requests
uv add --dev pytest
uv sync
uv lock
```

Tip: `uv add <pkg>==<version>` pins a specific version; omit to allow the latest compatible version.


In [ ]:
# Add dependencies and lock
uv add requests | cat
uv add --dev pytest | cat
uv sync | cat
uv lock | cat


## 3) Create a mini package and a CLI
We will create a simple package `mini_uv_project` with a module and a console entrypoint using `pyproject.toml` `[project.scripts]`.

Structure (will be created by the commands below):

- `mini_uv_project/`
  - `__init__.py`
  - `cli.py`

Entrypoint will be `mini-hello`.

Commands:


In [ ]:
%%writefile mini_uv_project/__init__.py
__version__ = '0.1.0'


In [ ]:
%%writefile mini_uv_project/cli.py
from __future__ import annotations
import sys

def main() -> int:
    name = sys.argv[1] if len(sys.argv) > 1 else "world"
    print(f"Hello, {name} 👋 (from mini_uv_project)")
    return 0

if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile -a pyproject.toml
[project.scripts]
mini-hello = "mini_uv_project.cli:main"


In [ ]:
# Install the project and run CLI via CLI commands
uv pip install -e .
uv run mini-hello
uv run mini-hello Student


## 4) Testing and linting
We'll add `pytest` for tests (already installed as dev dependency) and `ruff` for linting, and create a basic test to validate our CLI.

Commands:

```bash
uv add --dev ruff
uv run pytest -q
uv run ruff check .
```

We'll generate a minimal test file below and run both tools.


In [ ]:
# Ensure tests directory exists
mkdir -p tests

In [ ]:
%%writefile tests/test_cli.py
from mini_uv_project.cli import main


def test_main_runs(capsys):
    assert main() == 0
    captured = capsys.readouterr()
    assert "Hello, world" in captured.out


In [ ]:
# Add ruff and run tests/lint via CLI
uv add --dev ruff
uv run pytest -q
uv run ruff check .


## 5) Build and manage cache
Build source and wheel distributions with `uv build`. Use `uv cache` to inspect or clear cache if needed.

Commands:

```bash
uv build
uv cache dir
uv cache clear  # optional
```

We'll run them below.


In [ ]:
# Build distributions and show cache directory
uv build | cat
uv cache dir | cat
# Optionally clear cache (uncomment to run)
# uv cache clear | cat


## Wrap-up and next steps
You have:

- Initialized a project with `uv init`
- Added and locked dependencies with `uv add`, `uv sync`, `uv lock`
- Created a small package and a CLI entrypoint via `[project.scripts]`
- Added tests with `pytest` and linted with `ruff`
- Built distributions and inspected cache

Next ideas:
- Add more CLI commands and subcommands (e.g., `argparse`, `typer`)
- Configure `ruff` and `pytest` options in `pyproject.toml`
- Set up CI to run `uv run pytest` and `uv run ruff`
- Publish to an internal index or PyPI using your build artifacts in `dist/`
